# Conferência de vínculos Mercado Livre
Fotografia somente de leitura da conta CHGACDFBE12458 em 23/10/2026. Anúncio é uma publicação, produto é um registro local, e User Product é o agrupamento remoto. Não representam unidades físicas.
A coleta consultou GET /users/{seller}/items/search, GET /items/{id}?include_attributes=all, GET /user-products/{id} e products/channelListings/catalogItems da empresa no Firestore. Credenciais não estão neste arquivo. Candidatos por nome não confirmam correspondência de material, cor, dimensão ou kit.

In [1]:
import json
from pathlib import Path
from collections import Counter, defaultdict
snapshot = json.loads(Path("snapshot.json").read_text())
items, products = snapshot["items"], snapshot["products"]
assert len(items) == len({item["id"] for item in items}) == snapshot["expectedItems"]
assert all(call["status"] == 200 for call in snapshot["calls"])
assert all(up["httpStatus"] == 200 for up in snapshot["userProducts"])
missing_sku = sum(not (p["baseSku"] or "").strip() for p in products)
ml_with_sku = sum(bool(i["sku"] or i["sellerCustomField"] or any(v["sku"] or v["sellerCustomField"] for v in i["variations"])) for i in items)
groups = defaultdict(set)
for item in items:
    if item["userProductId"]:
        groups[item["userProductId"]].add(item["id"])
    for variation in item["variations"]:
        if variation["userProductId"]:
            groups[variation["userProductId"]].add(item["id"])
result = {"anuncios": len(items), "status": dict(Counter(i["status"] for i in items)),
          "produtos_lablayer": len(products), "produtos_sem_sku_base": missing_sku,
          "anuncios_com_sku": ml_with_sku, "vinculos_registrados": snapshot["summaries"]["channelListingCount"],
          "user_products": len(groups), "user_products_em_multiplos_anuncios": sum(len(v) > 1 for v in groups.values())}
print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "anuncios": 44,
  "status": {
    "closed": 1,
    "paused": 35,
    "active": 8
  },
  "produtos_lablayer": 133,
  "produtos_sem_sku_base": 131,
  "anuncios_com_sku": 0,
  "vinculos_registrados": 0,
  "user_products": 41,
  "user_products_em_multiplos_anuncios": 4
}
